In [1]:
from pathlib import Path

import polars as pl
import plotly.graph_objects as go
import plotly.express as px

In [2]:
collected_vehicle_data_path = Path("..", "raw_data", "collected-data", "collected_vehicle_data.csv")

# Research Question: What is the distribution of vehicle model makes in the target population?

In [3]:
obs = pl.scan_csv(source=collected_vehicle_data_path)

In [4]:
makes = (obs
    .group_by("make")
    .agg(pl.len().alias("count"))
    .sort("count", descending=False)
)

n = makes.select(pl.col("count").sum()).collect().item()

In [5]:
uncommon_makes = (makes
    .filter(pl.col("count") < 5)
)

The uncommon makes with observed counts less than 5 were combined into an "other" category.  These uncommon makes are shown in @tbl-unc.

In [6]:
#| tbl-cap: Uncommon Makes in Sample
#| label: tbl-unc
#| tbl-align: center
with pl.Config(
    tbl_hide_column_data_types=True,
    tbl_hide_dataframe_shape=True,
    set_tbl_rows=-1
):
    display(uncommon_makes.collect())

make,count
"""Land Rover""",1
"""Scion""",1
"""Pontiac""",1
"""Cadillac""",1
"""Saturn""",1
"""Infiniti""",1
"""Genesis""",1
"""Alfa Romeo""",1
"""Mini""",2
"""Suzuki""",3


In [7]:
common_makes = (makes
    .filter(pl.col("count") >= 5)
)

In [8]:
makes_2 = (common_makes
    .collect()
    .vstack(
        (uncommon_makes
            .select(pl.lit("other").alias("make"), pl.col("count").sum())
            .collect()
        )
    )
    .sort("count", descending=False)
    .lazy()
)

In [9]:
#| label: fig-abund
x = makes_2.select(pl.Expr.log10(pl.col("count")/n + 1)).collect().to_series()
y = makes_2.select("make").collect().to_series()
color_discrete_sequence = ["rgb(55, 90, 127)"]*len(y)
color_discrete_sequence[9] = "rgb(106, 74, 113)"

fig = go.Figure(
    data=go.Bar(
        x=x,
        y=y,
        orientation="h",
        marker_color=color_discrete_sequence,
        hoverinfo="text",
        hovertext=makes_2.select((pl.lit("n = ") + pl.col("count").cast(pl.Utf8))).collect().to_series()
    )
)

fig.update_layout(
    title="Transformed Abundances in Sample",
    xaxis=dict(title="log10(1 + p) where p is the proportion in the sample"),
    yaxis=dict(title=None)
)
fig.show()